# Caracterización de redes urbanas de Colombia

Análisis **solo Colombia** (140 ciudades) a partir del dataset ensamblado
(`dataset/dataset_ciudades_completo.csv`). Incluye **curaduría de calidad**, análisis
morfológico de la red, efecto de la **altitud/topografía** y el **tamaño**, familias de
ciudades (clustering) y **accesibilidad peatonal** por región.

Ventana por ciudad: disco de 6 km de radio (~113 km²), red peatonal de OSM, grafo
topológicamente simplificado. Todas las ciudades comparten la misma extensión, así que la
densidad de intersecciones es directamente comparable.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, matplotlib.cm as cm
from scipy.stats import spearmanr, kruskal
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from pathlib import Path
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.25,
                     'axes.spines.top':False,'axes.spines.right':False})
FIG=Path('figuras_colombia'); FIG.mkdir(exist_ok=True)

d=pd.read_csv('dataset/dataset_ciudades_completo.csv')
co=d[d.pais=='Colombia'].copy()
AREA=np.pi*6**2
co['dens']=co['n_nodos']/AREA               # intersecciones por km² (ventana igual para todas)

def region(x):
    for k,n in [('Insular','Insular'),('Pacífico','Pacífica'),('Amazon','Amazonía'),
                ('Orinoq','Orinoquía'),('Llanos','Orinoquía'),('Caribe','Caribe'),('Andina','Andina')]:
        if k in x: return n
    return 'otro'
co['region']=co['region_geografica'].map(region)
print('Ciudades Colombia:', len(co))
print('Por región:', co['region'].value_counts().to_dict())

## 1. Curaduría de calidad

La cobertura de OpenStreetMap varía por ciudad. Definimos un **tier de calidad** según el
tamaño de la red descargada y la cantidad de POIs, para saber en qué ciudades confiar
(sobre todo para accesibilidad):

- **A – confiable**: red ≥1500 nodos y ≥50 POIs.
- **B – media**: utilizable con cautela.
- **C – limitada**: red <800 nodos, o <3 servicios con datos, o <20 POIs → accesibilidad poco fiable.

In [ ]:
def tier(r):
    if r.n_nodos<800 or r.servicios_con_datos<3 or r.n_pois_total<20: return 'C_limitada'
    if r.n_nodos<1500 or r.n_pois_total<50: return 'B_media'
    return 'A_confiable'
co['calidad']=co.apply(tier,axis=1)
print(co['calidad'].value_counts().to_string())

print('\n— Ciudades tier C (cobertura limitada; tratar aparte) —')
print(co[co.calidad=='C_limitada'][['ciudad','estado_departamento','n_nodos','n_pois_total','servicios_con_datos']]
      .sort_values('n_nodos').to_string(index=False))

# Aviso: en municipios muy rurales la 'poblacion' es municipal (incluye campo disperso),
# así que no equivale al tamaño de la mancha urbana (p. ej. Uribia, Tumaco, Turbo).
co['nodos_por_1000hab']=co['n_nodos']/(co['poblacion']/1000)
print('\nPoblación municipal grande pero red urbana pequeña (rural-disperso):')
print(co[(co.poblacion>150000)&(co.n_nodos<5000)][['ciudad','poblacion','n_nodos']].to_string(index=False))

# Guardar dataset curado (con region y calidad)
co.to_csv('dataset/colombia_curado.csv', index=False)
print('\n✓ guardado dataset/colombia_curado.csv')

# Subconjunto de trabajo: excluye tier C para las métricas sensibles
sub=co[co.calidad!='C_limitada'].copy()
conf=co[co.calidad=='A_confiable'].copy()
print('sub (A+B):',len(sub),'| conf (A):',len(conf))

## 2. Distribuciones de la forma de la red

In [ ]:
fig,axs=plt.subplots(1,3,figsize=(13,3.6))
for ax,c,t in zip(axs,['dens','grado_promedio','circuity_avg'],
                  ['Densidad intersecciones/km²','Grado promedio','Circuidad (1=recto)']):
    ax.hist(sub[c],bins=25,color='#2471A3',alpha=0.8,edgecolor='white')
    ax.axvline(sub[c].median(),color='red',ls='--',lw=1.5)
    ax.set_title(t); ax.set_ylabel('nº ciudades')
plt.tight_layout(); plt.savefig(FIG/'c1_distribuciones.png',bbox_inches='tight'); plt.show()
print(sub[['dens','grado_promedio','circuity_avg','clustering_prom','long_prom_arista_m']].describe().round(2).to_string())

## 3. La topografía manda: altitud vs forma de la red

In [ ]:
print('Spearman (Colombia, tier A+B, n=%d):'%len(sub))
for c,lbl in [('circuity_avg','circuidad'),('grado_promedio','grado'),('dens','densidad'),('clustering_prom','clustering')]:
    r,p=spearmanr(sub['altitud_m'],sub[c]); print(f'  altitud vs {lbl:10s}: rho={r:+.2f}  p={p:.1e}')

fig,ax=plt.subplots(figsize=(7,4.6))
colreg={'Andina':'#8E44AD','Caribe':'#E67E22','Pacífica':'#16A085','Orinoquía':'#2980B9','Amazonía':'#27AE60','Insular':'#C0392B'}
for reg,g in sub.groupby('region'):
    ax.scatter(g['altitud_m'],g['circuity_avg'],s=40,alpha=0.8,label=reg,color=colreg.get(reg,'gray'),edgecolor='white')
z=np.polyfit(sub['altitud_m'],sub['circuity_avg'],1); xs=np.linspace(0,sub.altitud_m.max(),50)
ax.plot(xs,np.poly1d(z)(xs),'k--',lw=1.5)
r,_=spearmanr(sub['altitud_m'],sub['circuity_avg'])
ax.set_xlabel('Altitud (m)'); ax.set_ylabel('Circuidad media')
ax.set_title(f'Altitud vs sinuosidad de la red (ρ={r:.2f})'); ax.legend(fontsize=8,frameon=False)
plt.tight_layout(); plt.savefig(FIG/'c2_altitud.png',bbox_inches='tight'); plt.show()

## 4. Un solo eje: damero ↔ orgánico

In [ ]:
r,p=spearmanr(sub['circuity_avg'],sub['grado_promedio'])
print(f'circuidad vs grado: rho={r:.2f} (p={p:.1e}) -> un eje continuo grid<->orgánico')
print('\nMÁS GRID (circuidad baja):')
print(conf.nsmallest(7,'circuity_avg')[['ciudad','region','circuity_avg','grado_promedio','altitud_m']].to_string(index=False))
print('\nMÁS ORGÁNICA (circuidad alta):')
print(conf.nlargest(7,'circuity_avg')[['ciudad','region','circuity_avg','grado_promedio','altitud_m']].to_string(index=False))

## 5. Diferencias por región

In [ ]:
tab=sub.groupby('region').agg(n=('ciudad','size'),dens=('dens','median'),
     grado=('grado_promedio','median'),circ=('circuity_avg','median'),
     clust=('clustering_prom','median')).round(3).sort_values('dens',ascending=False)
print(tab.to_string())
# Kruskal-Wallis: ¿difiere la forma entre regiones?
for c in ['dens','grado_promedio','circuity_avg']:
    grupos=[g[c].values for _,g in sub.groupby('region') if len(g)>=4]
    H,p=kruskal(*grupos); print(f'  Kruskal {c}: H={H:.1f} p={p:.1e}')

fig,ax=plt.subplots(figsize=(7.5,4.2))
order=tab.index.tolist()
data=[sub[sub.region==r]['circuity_avg'] for r in order]
bp=ax.boxplot(data,labels=order,patch_artist=True,showfliers=False,medianprops=dict(color='black'))
for b in bp['boxes']: b.set(facecolor='#5DADE2',alpha=0.7)
ax.set_ylabel('Circuidad media'); ax.set_title('Sinuosidad de la red por región (Caribe=grid, Andina=orgánica)')
plt.tight_layout(); plt.savefig(FIG/'c3_region.png',bbox_inches='tight'); plt.show()

## 6. Tamaño y densidad

In [ ]:
r,p=spearmanr(sub['poblacion'],sub['dens'])
fig,ax=plt.subplots(figsize=(7,4.4))
for reg,g in sub.groupby('region'):
    ax.scatter(g['poblacion'],g['dens'],s=40,alpha=0.8,label=reg,color=colreg.get(reg,'gray'),edgecolor='white')
ax.set_xscale('log'); ax.set_xlabel('Población municipal (log)'); ax.set_ylabel('Densidad intersecciones/km²')
ax.set_title(f'Tamaño vs densidad de red (ρ={r:.2f})'); ax.legend(fontsize=8,frameon=False)
for _,x in sub.nlargest(4,'dens').iterrows(): ax.annotate(x['ciudad'],(x['poblacion'],x['dens']),fontsize=7)
plt.tight_layout(); plt.savefig(FIG/'c4_poblacion.png',bbox_inches='tight'); plt.show()

## 7. Familias morfológicas (clustering)

In [ ]:
feat=['grado_promedio','circuity_avg','clustering_prom','long_prom_arista_m','dens']
X=(sub[feat]-sub[feat].mean())/sub[feat].std()
Z=linkage(X,'ward'); sub['cluster']=fcluster(Z,4,'maxclust')
fig,ax=plt.subplots(figsize=(13,4))
dendrogram(Z,labels=sub['ciudad'].values,ax=ax,leaf_rotation=90,color_threshold=0.6*Z[:,2].max())
ax.set_title('Familias morfológicas de ciudades colombianas (Ward)'); ax.set_ylabel('distancia')
for t in ax.get_xticklabels(): t.set_fontsize(5.5)
plt.tight_layout(); plt.savefig(FIG/'c5_dendrograma.png',bbox_inches='tight'); plt.show()

for cl in sorted(sub.cluster.unique()):
    g=sub[sub.cluster==cl]
    print(f'Familia {cl} (n={len(g)}): dens={g.dens.median():.0f} grado={g.grado_promedio.median():.2f} '
          f'circ={g.circuity_avg.median():.3f} alt={g.altitud_m.median():.0f}m')
    print('   regiones:',dict(g.region.value_counts()),'| ej:',', '.join(g.nlargest(min(5,len(g)),'poblacion').ciudad),'\n')

## 8. Accesibilidad peatonal por región

In [ ]:
serv=['hospital','clinica','farmacia','escuela','mercado','parque','policia','bomberos']
acc=sub.groupby('region')[[f'dist_{s}_median' for s in serv]].median().round(0)
acc.columns=[c.replace('dist_','').replace('_median','') for c in acc.columns]
print('Distancia mediana al servicio más cercano (m), por región:')
print(acc.to_string())

fig,ax=plt.subplots(figsize=(9,4))
sserv=['hospital','farmacia','escuela','mercado']
order=['Andina','Caribe','Pacífica','Orinoquía','Amazonía']
x=np.arange(len(sserv)); w=0.16
for i,reg in enumerate(order):
    vals=[sub[sub.region==reg][f'dist_{s}_median'].median() for s in sserv]
    ax.bar(x+i*w,vals,w,label=reg,color=colreg.get(reg,'gray'),alpha=0.85)
ax.set_xticks(x+2*w); ax.set_xticklabels([s.capitalize() for s in sserv])
ax.set_ylabel('Distancia mediana (m)'); ax.set_title('Accesibilidad por región (menor = mejor)')
ax.legend(fontsize=8,frameon=False,ncol=5)
plt.tight_layout(); plt.savefig(FIG/'c6_accesibilidad.png',bbox_inches='tight'); plt.show()

## 9. Inequidad del acceso

In [ ]:
ineq=sub.groupby('region')[[f'dist_{s}_cv' for s in ['farmacia','mercado','escuela']]].median().round(2)
ineq.columns=['CV_farmacia','CV_mercado','CV_escuela']
print('Coeficiente de variación de la distancia (mayor = acceso más desigual):')
print(ineq.sort_values('CV_farmacia',ascending=False).to_string())

## 10. Síntesis

Ver el mensaje de análisis. En resumen: la **altitud** es el gran organizador de la forma de la
red en Colombia (ρ≈+0.72 con la sinuosidad); existe un **eje continuo damero↔orgánico**
(Caribe plano y reticulado vs Andes sinuosos); el **tamaño** predice la densidad; y la
**accesibilidad** es mejor en el Caribe y peor en Pacífico/Amazonía/Insular, con la salvedad
de la cobertura desigual de OSM (tier C).